## Cosmic Web - cuda.compute - Iterators - SOLUTION

[cuda.compute](https://nvidia.github.io/cccl/unstable/python/compute/index.html) is the Python library of the CUDA Core Compute Libraries (CCCL): device-wide reductions, scans, sorts and histograms, each running kernels of CUB, one of CCCL's C++ libraries. The kernels read their input through *iterators* (CCCL's fancy iterators): each computes its elements on demand as the kernel reads them. This is the *algorithm model*. In this notebook we write the stencil as iterators and hand them to the histogram algorithm.

**SOLUTION**

### Table of Contents

1. [Setup](#1-setup)
2. [Iterators](#2-iterators)
3. [Histogram](#3-histogram)
4. [Trace](#4-trace)
5. [Limitations and recap](#5-limitations-and-recap)

---

### 1. Setup

We use the binned field as one flat array, so a voxel is one index $t$. cuda.compute reads CuPy arrays directly.

In [1]:
import importlib.metadata

import numpy as np
import cupy as cp
import cuda.compute as comp

import cosmic_core as cw

N, K = cw.setup('cccl', f'cuda-cccl {importlib.metadata.version("cuda-cccl")}')

N = 512^3, K = 128: 512 MiB of int32 on NVIDIA GeForce RTX 4080 SUPER; cuda-cccl 1.1.1


### 2. Iterators

A cell's birth is the minimum over its corners, and a corner is the field read at a neighbour's index. Four iterators express that, reading the corners on demand. An *op* is a plain Python function that the library compiles for the device. It may close over integers such as `n`, and arrays reach it only through iterators.

The vertex and the three edges are written. The TODO adds the corner sets of the three faces and the cube.

---

**Quick Docs**

- `comp.CountingIterator(np.int32(start))`: `start, start + 1, ...`, here the voxel numbers $t$.
- `comp.TransformIterator(underlying, op)`: `op(x)` for each element as it is read, here the flat index of a neighbour.
- `comp.PermutationIterator(values, indices)`: `values[i]` for each `i` the index iterator yields, here one corner.
- `comp.ZipIterator(*iterators)`: a tuple per position, one element from each; a transform over it takes the minimum of the corners.

In [2]:
def neighbour_index(di, dj, dk, n):
    '''The op mapping voxel t to the flat index of its corner at offset (di, dj, dk), periodic.'''
    nn = n * n

    def op(t):
        i = t // nn
        j = (t - i * nn) // n
        k = t - i * nn - j * n
        # The offsets are 0 or 1, so one compare per axis wraps the index.
        i1 = i + di if i + di < n else 0
        j1 = j + dj if j + dj < n else 0
        k1 = k + dk if k + dk < n else 0
        return i1 * nn + j1 * n + k1
    return op


def corner(q, offset, n):
    '''The field read at every voxel's corner at `offset`: a permutation of q by the neighbour index.'''
    index = comp.TransformIterator(comp.CountingIterator(np.int32(0)), neighbour_index(*offset, n))
    return comp.PermutationIterator(q, index)


# One op per corner count: the zip tuple's length is fixed per cell type.
def min2(c): return min(c[0], c[1])
def min4(c): return min(min(c[0], c[1]), min(c[2], c[3]))
def min8(c): return min(min(min(c[0], c[1]), min(c[2], c[3])), min(min(c[4], c[5]), min(c[6], c[7])))

# Each cell type: its sign in the histogram and the offsets of its corners from the voxel.
CELLS = {
    'v':   (+1, [(0, 0, 0)]),
    'ex':  (-1, [(0, 0, 0), (1, 0, 0)]),
    'ey':  (-1, [(0, 0, 0), (0, 1, 0)]),
    'ez':  (-1, [(0, 0, 0), (0, 0, 1)]),
    'fxy': (+1, [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0)]),
    'fxz': (+1, [(0, 0, 0), (1, 0, 0), (0, 0, 1), (1, 0, 1)]),
    'fyz': (+1, [(0, 0, 0), (0, 1, 0), (0, 0, 1), (0, 1, 1)]),
    'c':   (-1, [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0), (0, 0, 1), (1, 0, 1), (0, 1, 1), (1, 1, 1)]),
}


def births(q, n, corners):
    '''An iterator over the births of one cell type: the minimum of the field over the cell's corners.'''
    if len(corners) == 1:
        return q
    parts = [q] + [corner(q, c, n) for c in corners[1:]]
    return comp.TransformIterator(comp.ZipIterator(*parts), {2: min2, 4: min4, 8: min8}[len(corners)])

### 3. Histogram

`histogram_even` takes bin boundaries, its `levels`; with $K + 1$ of them at $0, 1, \dots, K$ each bin covers one integer, so bin $b$ counts the births equal to $b$. The algorithm takes one input and fills one histogram per call, so the eight cell types need eight calls: eight passes over the field. The TODO writes the `histogram_even` call. The loop, the signed sum and the runner are written.

---

**Quick Docs**

- `comp.histogram_even(d_samples, d_histogram, num_output_levels, lower_level, upper_level, num_samples, stream)`: counts `num_samples` elements of `d_samples` into `num_output_levels - 1` bins between the two levels; every argument is keyword-only.
- `cp.cuda.get_current_stream()`: CuPy's current stream; passing it orders the library's kernels after CuPy's.

In [3]:
def ecc_hist_cccl(samples, hists, n):
    '''Signed histogram of births (host int64, length K): one histogram_even per cell type into hists[name].'''
    stream = cp.cuda.get_current_stream()
    for name, it in samples.items():
        comp.histogram_even(d_samples=it, d_histogram=hists[name], num_output_levels=K + 1,
                            lower_level=np.int32(0), upper_level=np.int32(K), num_samples=n ** 3, stream=stream)
    # The signed sum of the eight counts, on the host, is the token histogram.
    return sum(CELLS[name][0] * cp.asnumpy(hists[name]).astype(np.int64) for name in hists)


def cccl_state(n):
    '''The births iterators and the histogram buffers for size n.'''
    q = cw.field(n).ravel()
    return ({name: births(q, n, corners) for name, (_, corners) in CELLS.items()},
            {name: cp.zeros(K, dtype=cp.int32) for name in CELLS})


STATE = cw.by_size(cccl_state)


def run_cccl(n):
    return ecc_hist_cccl(*STATE(n), n)

Now let's make sure it works.

In [4]:
warm = cw.measure('cccl', run_cccl, N, cold_note='incl. the library building its kernels')

[cccl] N=512^3 K=128 | cold  12793.4 ms (incl. the library building its kernels) | warm     13.7 ms = 9.782 Gvoxel/s | max|Δχ| 0 | Σ 0 | PASS


### 4. Trace

How many kernels does the library launch per histogram? The bytes model counts one read of the field per cell type.

In [5]:
trace = cw.trace_call('cccl', run_cccl, N, [neighbour_index, corner, min2, min4, min8, births, ecc_hist_cccl, cccl_state],
                      header='import cuda.compute as comp', state=[f'CELLS = {CELLS!r}', STATE])
sweeps = [d for _, d, name, kind in trace['rows'] if kind == 'kernel' and 'Sweep' in name]  # CUB runs an init kernel and a sweep kernel per histogram
print('sweep kernel per cell type: ' + ', '.join(f'{name} {d * 1e3:.2f} ms' for name, d in zip(CELLS, sweeps)))
cw.roof_report('cccl', warm)
cw.save_sweep('cccl', run_cccl)

one call at N = 512^3: 16 kernels, 8 memcpys, 0 memsets
sweep kernel per cell type: v 0.78 ms, ex 1.12 ms, ey 0.81 ms, ez 0.82 ms, fxy 1.56 ms, fxz 1.56 ms, fyz 1.55 ms, c 5.47 ms
operation                                     count  MiB each  MiB total
histogram pass (one read of the field each)       8   512.000     4096.0
model 4096 MiB -> 6.84 ms at 628 GB/s; measured warm 13.72 ms = 2.00 x the roof


  N=  64^3  warm      0.36 ms    0.735 Gvoxel/s  max|Δχ| 0  Σ 0


  N= 128^3  warm      0.40 ms    5.297 Gvoxel/s  max|Δχ| 0  Σ 0


  N= 256^3  warm      1.56 ms   10.771 Gvoxel/s  max|Δχ| 0  Σ 0
  N= 512^3  warm     13.81 ms    9.721 Gvoxel/s  max|Δχ| 0  Σ 0


- Each histogram runs two library kernels: an initialisation that zeroes the counters and a sweep that counts. The iterators are evaluated inside the sweep, so the trace shows only library kernels.
- The memcpys are ours: one device-to-host copy per histogram for the signed sum.
- The sweep time grows with the number of corners: the model counts one read of the field per pass, while the kernel reads the field once per corner.

Takeaways:

- The stencil is evaluated on demand: the library's kernel reads the corners it needs through the iterators as it runs.
- The algorithm fills one histogram per call, so eight cell types cost eight passes over the field, where a kernel of our own could make one.

### 5. Limitations and recap

The library's kernels are fixed: the library sets their thread mapping and their number of passes.

Recap:

- Iterators described the stencil and `histogram_even` consumed it, one call per cell type.

Next: `94__cosmic_web__cppjit__cuda_kernel.ipynb` compiles CUDA C++ in the session: CUB's histogram from C++ first, then one kernel for all eight passes.